In [ ]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint
from IPython.display import Image, Markdown, display

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

from data.share_data import load_share_data
from scripts.misocp import plot_misocp_outputs, run_global_misocp_ev
from utils.records import compute_ev_agent_cost_summary, compute_ev_cost_summary
from utils.run_artifacts import load_experiment_context


In [ ]:
run_dir = Path("artifacts/runs/20260512_220936_494b4c57")
cfg, run_dir = load_experiment_context(run_dir)
cfg.env.ev_enabled = True
cfg.model.action_dim = 3
cfg.env.ev_departure_constraint_mode = "emergency"
cfg.env.ev_hard_projection_enabled = False
cfg.env.ev_emergency_charging_enabled = True
cfg.env.ev_emergency_window_hours = 4.0
cfg.env.ev_emergency_strategy = "required_power"
cfg.env.ev_capacity_kwh = (60.0, 60.0, 60.0)
cfg.env.ev_soc_min = 0.10
cfg.env.ev_soc_max = 0.95
cfg.env.ev_arrival_soc = 0.15
cfg.env.ev_departure_soc_req = 0.90
cfg.env.ev_max_charge_kw = (11.0, 11.0, 11.0)
cfg.env.ev_efficiency = 0.95
cfg.env.ev_arrival_step = 72
cfg.env.ev_departure_step = 28
share_data = load_share_data(run_dir / "share_data", cfg)
display(Markdown("# MADRL_ESS MISOCP + EV Emergency Charging"))
summary = {
    "run_dir": str(run_dir),
    "ev_enabled": bool(cfg.env.ev_enabled),
    "ev_departure_constraint_mode": cfg.env.ev_departure_constraint_mode,
    "action_dim": int(cfg.model.action_dim),
    "ev_capacity_kwh": cfg.env.ev_capacity_kwh,
    "ev_arrival_soc": float(cfg.env.ev_arrival_soc),
    "ev_departure_soc_req": float(cfg.env.ev_departure_soc_req),
    "ev_max_charge_kw": cfg.env.ev_max_charge_kw,
    "ev_arrival_step": int(cfg.env.ev_arrival_step),
    "ev_departure_step": int(cfg.env.ev_departure_step),
}
if cfg.env.ev_departure_constraint_mode == "emergency":
    summary["ev_emergency_window_hours"] = float(cfg.env.ev_emergency_window_hours)
    summary["ev_emergency_strategy"] = cfg.env.ev_emergency_strategy
display(pd.DataFrame([summary]))


In [ ]:
record = run_global_misocp_ev(cfg, run_dir, share_data, ev_mode="emergency")
display(Markdown("## Metrics"))
display(record["metrics_df"])
display(Markdown("## Agent Summary"))
display(record["rollout"].summary)
pprint({"record_dir": str(record["record_dir"])})

display(Markdown("## Total Cost Summary"))
cost_ts, cost_summary = compute_ev_cost_summary(record["rollout"], cfg)
agent_cost_summary = compute_ev_agent_cost_summary(record["rollout"], cfg)
display(cost_summary)
display(Markdown("### Cost by Agent"))
display(agent_cost_summary)


In [ ]:
figures = plot_misocp_outputs(record, run_dir)
_plot_specs = [
    ("1. Power balance", "misocp_power_balance"),
    ("2. Price", "misocp_price"),
    ("3. Battery power and SoC", "misocp_battery_soc"),
    ("4. Voltage", "misocp_voltage"),
    ("5. Net load", "misocp_net_load"),
    ("6. EV SOC", "misocp_ev_soc"),
    ("7. EV charging power", "misocp_ev_charge_kw"),
    ("8. EV departure SOC requirement check", "misocp_ev_departure_check"),
    ("9. Total cumulative cost", "misocp_total_cost"),
    ("10. Cost components per step", "misocp_cost_components"),
    ("11. Cumulative cost components", "misocp_cumulative_cost_components"),
]
for title, name in _plot_specs:
    path = run_dir / "figures" / f"{name}.png"
    if path.exists():
        print(title)
        display(Image(filename=str(path)))
        if name in figures:
            plt.close(figures[name])
    elif name in figures:
        print(title)
        display(figures[name])
        plt.close(figures[name])
    else:
        print(f"[missing] {title}: {path}")
